# PH-SHOWOA · Python/PyTorch CUDA SA-RCRS-GRASP Solver Benchmark (Commit d88da6e)

**Mục tiêu**: Chạy kiểm thử phiên bản commit `d88da6eb892e3ccc3f17d4fdcb024104d0ef516e` trên Kaggle (bản cho kết quả tối ưu vượt target trên 15 bộ benchmark).

**Thông số thử nghiệm**:
- `Compute Backend` = `cuda`
- `Architecture` = `python_cuda`
- `Init` = `sa_rcrs_grasp`
- `Popsize` = 32
- `Num Islands` = 4 (Mỗi đảo 8 cá thể)
- `Max-iteration` = 1000
- `Runs` = 30
- `Objective` = `lexicographic`


## Cell 1 – Chuẩn bị Repo (Commit d88da6eb892e3ccc3f17d4fdcb024104d0ef516e)

In [ ]:
import os

COMMIT_HASH = "d88da6eb892e3ccc3f17d4fdcb024104d0ef516e"
REPO_URL = "https://github.com/Welkie/ph-showoa.git"

# Kiểm tra xem code d88 đã có sẵn hay cần clone
if os.path.exists("src_python_gpu_SA_RCRS_GRASP"):
    print("[INFO] Đang chạy trực tiếp trong thư mục mã nguồn.")
elif os.path.exists("ph-showoa"):
    print(f"[INFO] Thư mục ph-showoa tồn tại. Cập nhật và checkout {COMMIT_HASH}...")
    !cd ph-showoa && git fetch origin && git checkout {COMMIT_HASH}
elif os.path.exists("ph-showoa-d88da6eb892e3ccc3f17d4fdcb024104d0ef516e"):
    print("[INFO] Tìm thấy thư mục ph-showoa-d88.")
else:
    print(f"[INFO] Tiến hành clone repo và checkout {COMMIT_HASH}...")
    !git clone {REPO_URL} ph-showoa && cd ph-showoa && git checkout {COMMIT_HASH}

print(f"[OK] Sẵn sàng thực thi với commit {COMMIT_HASH}")


## Cell 2 – Kiểm tra GPU & Môi trường PyTorch CUDA

In [ ]:
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU runtime is required: enable a CUDA accelerator before running.")
torch.cuda.set_device(0)
print("Device name:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))


## Cell 3 – Xác nhận Python/PyTorch CUDA Backend

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for the Python/PyTorch CUDA solver.")

torch.cuda.set_device(0)
print("Python/PyTorch CUDA backend ready")
print("Device:", torch.cuda.get_device_name(0))
print("Implementation: Pure PyTorch CUDA Tensors (d88)")


## Cell 4 – Batch runner Python/PyTorch CUDA SA-RCRS-GRASP (15 bộ benchmark, pop_size=32, runs=30)

In [ ]:
import glob
import os
import re
import subprocess
import sys
import time as T
from collections import deque

import pandas as pd

# 15 bài toán benchmark chuẩn theo bảng so sánh
DATASETS = [
    "rcdp1001", "rcdp5001", "rcdp5007", "rcdp5004", "rcdp101",
    "cdp103", "rcdp205", "rdp210", "rcdp207", "rcdp202",
    "rdp103", "cdp104", "cdp102", "rdp203", "rcdp104",
]

# Tự động phát hiện thư mục làm việc chứa repo d88
candidate_cwds = [
    "ph-showoa",
    "ph-showoa-d88da6eb892e3ccc3f17d4fdcb024104d0ef516e",
    ".",
    "..",
    "/kaggle/working/ph-showoa",
    "/kaggle/working/ph-showoa-d88da6eb892e3ccc3f17d4fdcb024104d0ef516e",
]
CWD = next((d for d in candidate_cwds if os.path.isdir(os.path.join(d, "src_python_gpu_SA_RCRS_GRASP"))), ".")
print(f"[INFO] CWD mã nguồn: {os.path.abspath(CWD)}")

DATASET_DIR = "/kaggle/input/datasets/keith1101/ph-showoa/Wang_Chen"
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "instances_benchmark_pop_size32_runs=30_iter1000.csv")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs_d88_gpu")

# Cấu hình tham số chuẩn của bản d88 (pop_size=32, 4 đảo, 30 runs, 1000 iter)
RUNS = 30
MAX_ITER = 1000
POP_SIZE = 32
NUM_ISLANDS = 4
INIT_MODE = "sa_rcrs_grasp"
COMPUTE_BACKEND = "cuda"
ARCHITECTURE = "python_cuda"
WORKERS = 1
RESUME = True
MAX_TIMEOUT_S = None
TAIL_LINES = 20
os.makedirs(LOG_DIR, exist_ok=True)

def find_file(name):
    target_names = [
        f"explicit_{name}.vrpsdptw",
        f"explicit_{name.lower()}.vrpsdptw",
        f"explicit_{name.upper()}.vrpsdptw",
        f"{name}.vrpsdptw",
        f"{name.lower()}.vrpsdptw",
    ]
    search_dirs = [
        os.path.join(CWD, "dataset"),
        DATASET_DIR,
        "/kaggle/input",
        ".",
    ]
    for target in target_names:
        for sdir in search_dirs:
            if not os.path.exists(sdir):
                continue
            direct = os.path.join(sdir, target)
            if os.path.isfile(direct):
                return direct
            matches = glob.glob(os.path.join(sdir, "**", target), recursive=True)
            if matches:
                return matches[0]
    all_files = glob.glob(f"**/*{name}*.vrpsdptw", recursive=True)
    if all_files:
        return all_files[0]
    all_kaggle = glob.glob(f"/kaggle/input/**/*{name}*.vrpsdptw", recursive=True)
    if all_kaggle:
        return all_kaggle[0]
    return None

def parse_output(text):
    if "Traceback (most recent call last)" in text or "RuntimeError:" in text:
        return None
    runs = re.search(r"Total (\d+) runs, total consumed ([\d.]+) sec", text)
    nv = re.search(r"(?:Vehicle count|vehicle \(route\) number):\s*(\d+)", text)
    cost = re.search(r"Total cost:\s*([\d.]+)", text)
    distance = re.search(r"Total distance:\s*([\d.]+)", text)
    if not (runs and nv and cost) or int(runs.group(1)) != RUNS:
        return None
    total_runs = int(runs.group(1))
    total_cost = float(cost.group(1))
    total_distance = float(distance.group(1)) if distance else total_cost - 2000.0 * int(nv.group(1))
    return {
        "best_NV": int(nv.group(1)),
        "best_TD": f"{total_distance:.4f}",
        "total_cost": f"{total_cost:.4f}",
        "avg_time_s": f"{float(runs.group(2)) / total_runs:.2f}",
        "total_runs": total_runs,
        "Status": "Success",
    }

def save_summary(rows):
    columns = ["Dataset", "best_NV", "best_TD", "total_cost", "avg_time_s", "wall_time", "total_runs", "Status", "Error"]
    frame = pd.DataFrame(rows)
    for column in columns:
        if column not in frame:
            frame[column] = "N/A"
    frame = frame[columns]
    frame.columns = ["Dataset", "Best NV", "Best TD", "Total Cost", "Avg/Run", "Wall Time", "Runs", "Status", "Error"]
    frame.to_csv(OUTPUT_CSV, index=False)

results = []
completed_names = set()
if RESUME and os.path.exists(OUTPUT_CSV):
    previous = pd.read_csv(OUTPUT_CSV)
    for _, row in previous.iterrows():
        if str(row.get("Status", "")).strip().lower() != "success":
            continue
        name = str(row.get("Dataset", "")).strip()
        results.append({
            "Dataset": name,
            "best_NV": row.get("Best NV", "N/A"),
            "best_TD": row.get("Best TD", "N/A"),
            "total_cost": row.get("Total Cost", "N/A"),
            "avg_time_s": row.get("Avg/Run", "N/A"),
            "wall_time": row.get("Wall Time", "N/A"),
            "total_runs": row.get("Runs", "N/A"),
            "Status": "Success",
            "Error": ""
        })
        completed_names.add(name)

for name in DATASETS:
    if name in completed_names:
        print(f"[RESUME] Bỏ qua bộ đã chạy thành công: {name}")
        continue
    problem = find_file(name)
    if problem is None:
        results.append({"Dataset": name, "Status": "File Not Found", "Error": "dataset file not found"})
        save_summary(results)
        print(f"[ERROR] Không tìm thấy file bộ dữ liệu: {name}")
        continue

    command = [
        sys.executable, "-u", "-m", "src_python_gpu_SA_RCRS_GRASP.main",
        "--problem", problem, "--compute_backend", COMPUTE_BACKEND,
        "--init", INIT_MODE, "--paper_flags", "--architecture", ARCHITECTURE,
        "--objective", "lexicographic", "--grasp_alpha_lo", "0.10",
        "--grasp_alpha_hi", "0.40", "--sa_iterations", "25",
        "--runs", str(RUNS), "--max_iter", str(MAX_ITER),
        "--pop_size", str(POP_SIZE), "--num_islands", str(NUM_ISLANDS),
        "--workers", str(WORKERS),
    ]
    print(f"\n[RUN] {name}: Python/PyTorch CUDA (d88) | pop_size={POP_SIZE}, islands={NUM_ISLANDS}, target runs={RUNS}")
    run_started = T.time()
    last_report = run_started
    lines = []
    tail = deque(maxlen=TAIL_LINES)
    timed_out = False
    process = subprocess.Popen(command, cwd=CWD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            lines.append(line)
            clean = line.rstrip()
            tail.append(clean)
            now = T.time()
            run_marker = re.search(r"Run (\d+).*", clean)
            total_marker = re.search(r"Total (\d+) runs, total consumed", clean)
            if run_marker and "Run " in clean:
                print(f"  [{now - run_started:.1f}s] {clean}", flush=True)
            elif total_marker:
                print(f"  [{now - run_started:.1f}s] {clean}", flush=True)
            elif now - last_report >= 20:
                print(f"  [{now - run_started:.0f}s] running... last: {clean[:100]}", flush=True)
                last_report = now
            if MAX_TIMEOUT_S and now - run_started > MAX_TIMEOUT_S:
                timed_out = True
                process.kill()
                break
        process.wait()
    except Exception:
        process.kill()
        process.wait()
        raise

    wall = T.time() - run_started
    output = "".join(lines)
    with open(os.path.join(LOG_DIR, f"{name}.log"), "w", encoding="utf-8") as log:
        log.write(output)
    return_code = process.returncode
    parsed = parse_output(output) if return_code == 0 and not timed_out else None
    if parsed is None:
        error_lines = [line.strip() for line in lines if "error" in line.lower() or "traceback" in line.lower()]
        row = {"Dataset": name, "Status": "Timeout" if timed_out else "Failed", "Error": " | ".join(error_lines[-3:]) or f"Python process exit code {return_code}", "wall_time": f"{wall:.1f}s"}
        print(f"[FAILED] {name}: {row['Error']}")
    else:
        row = parsed | {"Dataset": name, "wall_time": f"{wall:.1f}s", "Error": ""}
        completed_names.add(name)
        print(f"[OK] {name}: hoàn thành {parsed['total_runs']}/{RUNS} runs | NV={row['best_NV']} TD={row['best_TD']}")
    results.append(row)
    save_summary(results)

print(f"\n[HOÀN TẤT] Bảng kết quả đã được ghi vào: {OUTPUT_CSV}")


## Cell 5 – Bảng tổng hợp kết quả Benchmark & Xuất CSV

In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "instances_benchmark_pop_size32_runs=30_iter1000.csv")

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print("BẢNG TỔNG HỢP KẾT QUẢ BENCHMARK BẢN D88 (pop_size=32, runs=30, max_iter=1000):\n")
    print(df.to_string(index=False))
    print(f"\n[OK] Đã lưu bảng kết quả tại: {OUTPUT_CSV}")
else:
    print(f"Chưa tìm thấy file kết quả {OUTPUT_CSV}.")
